# Notebook 02 - Exploring Public Datasets

**Topic:** Data Collection Strategies  
**Task:** Download the Titanic or Iris dataset, run a dataset audit, and describe 3 meaningful findings

---

## What you will learn

- How to load public datasets from URLs and from `scikit-learn`
- How to perform a systematic dataset audit (shape, types, missing values, duplicates)
- How to compute and interpret descriptive statistics
- How to identify and document real findings from raw data

---

## Prerequisites

```bash
pip install pandas numpy scikit-learn matplotlib seaborn
```

---

## Section 1 - Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.datasets import load_iris

os.makedirs("output", exist_ok=True)

# Plot styling
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print("Libraries imported successfully.")

---

## Section 2 - The Dataset Audit Function

Before doing anything else with a dataset, you should always run a structured audit. This gives you a reliable picture of what you are working with before making any assumptions.

The function below is a reusable audit tool you should keep in your workflow.

In [ ]:
def audit_dataset(df, name="Dataset"):
    """
    Print a structured audit report for a DataFrame.

    Parameters
    ----------
    df   : pd.DataFrame
    name : str - Label shown in the report header
    """
    sep = "-" * 55

    print(f"\n{'=' * 55}")
    print(f"  AUDIT REPORT: {name}")
    print(f"{'=' * 55}")

    # Shape
    print(f"\n[1] Shape")
    print(f"    Rows    : {df.shape[0]}")
    print(f"    Columns : {df.shape[1]}")

    # Column types
    print(f"\n[2] Column data types")
    for col, dtype in df.dtypes.items():
        print(f"    {col:<30} {dtype}")

    # Missing values
    print(f"\n[3] Missing values")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({"count": missing, "pct": missing_pct})
    missing_df = missing_df[missing_df["count"] > 0]
    if missing_df.empty:
        print("    No missing values.")
    else:
        for col, row in missing_df.iterrows():
            print(f"    {col:<30} {int(row['count'])} missing ({row['pct']}%)")

    # Duplicates
    dupes = df.duplicated().sum()
    print(f"\n[4] Duplicate rows: {dupes}")

    # Numeric summary
    num_cols = df.select_dtypes(include="number").columns.tolist()
    if num_cols:
        print(f"\n[5] Numeric column summary")
        print(df[num_cols].describe().round(2).to_string())

    # Categorical columns
    cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    if cat_cols:
        print(f"\n[6] Categorical column cardinality")
        for col in cat_cols:
            n_unique = df[col].nunique()
            top = df[col].value_counts().index[0] if n_unique > 0 else "N/A"
            print(f"    {col:<30} {n_unique} unique values  |  most common: '{top}'")

    print(f"\n{'=' * 55}\n")


print("audit_dataset() is ready.")

---

## Section 3 - Dataset A: Titanic

### 3.1 Load the dataset

The Titanic dataset records passengers on the RMS Titanic and whether they survived. It is one of the most widely used datasets for learning classification and data analysis.

In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df_titanic = pd.read_csv(url)

print(f"Loaded Titanic dataset: {df_titanic.shape[0]} rows, {df_titanic.shape[1]} columns")
df_titanic.head()

### Column reference

| Column | Description |
|---|---|
| `Survived` | 0 = did not survive, 1 = survived |
| `Pclass` | Ticket class: 1 = first, 2 = second, 3 = third |
| `Sex` | Passenger sex |
| `Age` | Age in years |
| `SibSp` | Number of siblings or spouses aboard |
| `Parch` | Number of parents or children aboard |
| `Fare` | Ticket fare in British pounds |
| `Embarked` | Port of embarkation: C = Cherbourg, Q = Queenstown, S = Southampton |

### 3.2 Run the audit

In [ ]:
audit_dataset(df_titanic, name="Titanic")

### 3.3 Finding 1 - Survival rate was strongly linked to passenger class

First-class passengers had a dramatically higher survival rate than third-class passengers.

In [ ]:
survival_by_class = df_titanic.groupby("Pclass")["Survived"].mean().round(3)
print("Survival rate by passenger class:")
print(survival_by_class.rename({1: "1st class", 2: "2nd class", 3: "3rd class"}).to_string())
print("\nInterpretation:")
print("  1st class passengers survived at 3x the rate of 3rd class passengers.")
print("  This suggests proximity to lifeboats and priority access played a role.")

In [ ]:
ax = survival_by_class.plot(kind="bar", color=["#2a5fc1", "#5a8fd6", "#a0bce8"], width=0.5)
ax.set_xticklabels(["1st class", "2nd class", "3rd class"], rotation=0)
ax.set_ylabel("Survival rate")
ax.set_title("Titanic: Survival rate by passenger class")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("output/titanic_survival_by_class.png", dpi=120)
plt.show()

### 3.4 Finding 2 - Women survived at a far higher rate than men

The "women and children first" evacuation protocol is clearly visible in the data.

In [ ]:
survival_by_sex = df_titanic.groupby("Sex")["Survived"].mean().round(3)
counts_by_sex = df_titanic.groupby("Sex")["Survived"].count()

print("Survival rate by sex:")
print(survival_by_sex.to_string())
print("\nPassenger count by sex:")
print(counts_by_sex.to_string())

print("\nInterpretation:")
print(f"  Female survival rate: {survival_by_sex['female']:.1%}")
print(f"  Male survival rate  : {survival_by_sex['male']:.1%}")
print("  Females were roughly 4x more likely to survive than males.")

In [ ]:
fig, axes = plt.subplots(1, 2)

survival_by_sex.plot(kind="bar", ax=axes[0], color=["#e86d6d", "#5b8dd9"], width=0.45)
axes[0].set_title("Survival rate by sex")
axes[0].set_ylabel("Survival rate")
axes[0].set_xticklabels(["Female", "Male"], rotation=0)
axes[0].set_ylim(0, 1)

# Cross-tabulation: sex x class
cross = df_titanic.groupby(["Pclass", "Sex"])["Survived"].mean().unstack()
cross.plot(kind="bar", ax=axes[1], color=["#e86d6d", "#5b8dd9"], width=0.6)
axes[1].set_title("Survival rate by class and sex")
axes[1].set_ylabel("Survival rate")
axes[1].set_xticklabels(["1st", "2nd", "3rd"], rotation=0)
axes[1].set_ylim(0, 1)
axes[1].legend(["Female", "Male"])

plt.tight_layout()
plt.savefig("output/titanic_survival_by_sex.png", dpi=120)
plt.show()

### 3.5 Finding 3 - Age data is missing for 20% of passengers

Missing data is a real-world data quality issue. Nearly 1 in 5 passengers has no age recorded. This needs to be addressed before age can be used in any model.

In [ ]:
total = len(df_titanic)
missing_age = df_titanic["Age"].isnull().sum()
pct = missing_age / total * 100

print(f"Total passengers  : {total}")
print(f"Missing Age values: {missing_age} ({pct:.1f}%)")
print("\nAge distribution for passengers WITH age recorded:")
print(df_titanic["Age"].describe().round(2).to_string())

print("\nCommon strategies to handle missing age:")
print("  1. Drop rows with missing age (loses 20% of data)")
print("  2. Impute with median age (simple, conservative)")
print("  3. Impute using median age per Pclass and Sex (smarter)")
print("  4. Predict age using other columns (most sophisticated)")

In [ ]:
# Demonstrate strategy 3: group-based median imputation
df_imputed = df_titanic.copy()

df_imputed["Age"] = df_imputed.groupby(["Pclass", "Sex"])["Age"].transform(
    lambda x: x.fillna(x.median())
)

remaining_missing = df_imputed["Age"].isnull().sum()
print(f"Missing Age after group-based imputation: {remaining_missing}")

fig, axes = plt.subplots(1, 2)
df_titanic["Age"].dropna().plot(kind="hist", bins=30, ax=axes[0], color="#5b8dd9")
axes[0].set_title("Age distribution (original)")
axes[0].set_xlabel("Age")

df_imputed["Age"].plot(kind="hist", bins=30, ax=axes[1], color="#2a5fc1")
axes[1].set_title("Age distribution (after imputation)")
axes[1].set_xlabel("Age")

plt.tight_layout()
plt.savefig("output/titanic_age_imputation.png", dpi=120)
plt.show()

### 3.6 Save the cleaned dataset

In [ ]:
df_imputed.to_csv("output/titanic_cleaned.csv", index=False)
print("Saved titanic_cleaned.csv")

---

## Section 4 - Dataset B: Iris

### 4.1 Load the dataset

The Iris dataset contains measurements of 150 iris flowers across 3 species. It is the standard benchmark for multiclass classification.

In [ ]:
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

print(f"Loaded Iris dataset: {df_iris.shape[0]} rows, {df_iris.shape[1]} columns")
df_iris.head()

### 4.2 Run the audit

In [ ]:
audit_dataset(df_iris, name="Iris")

**Observation:** The Iris dataset is clean. No missing values, no duplicates, perfectly balanced classes (50 rows per species). This is rare in real-world data.

### 4.3 Finding 1 - Petal length alone can nearly separate the three species

In [ ]:
petal_stats = df_iris.groupby("species")["petal length (cm)"].describe().round(2)
print("Petal length by species:")
print(petal_stats.to_string())

print("\nInterpretation:")
print("  Setosa has petals < 2 cm.")
print("  Versicolor is in the 3-5 cm range.")
print("  Virginica has petals > 4.5 cm.")
print("  These ranges barely overlap, making petal length a powerful feature.")

In [ ]:
colors = {"setosa": "#e86d6d", "versicolor": "#5b8dd9", "virginica": "#52b788"}

fig, ax = plt.subplots()
for species, group in df_iris.groupby("species"):
    group["petal length (cm)"].plot(
        kind="kde", ax=ax, label=species, color=colors[species], linewidth=1.8
    )

ax.set_title("Iris: Petal length distribution by species")
ax.set_xlabel("Petal length (cm)")
ax.legend()
plt.tight_layout()
plt.savefig("output/iris_petal_length.png", dpi=120)
plt.show()

### 4.4 Finding 2 - Petal measurements are strongly correlated with each other

In [ ]:
corr = df_iris.drop(columns="species").corr().round(2)
print("Correlation matrix:")
print(corr.to_string())

print("\nInterpretation:")
print("  Petal length and petal width have correlation 0.96.")
print("  This high collinearity means using both in a linear model adds little new information.")

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="Blues",
    linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8}
)
ax.set_title("Iris: Feature correlation matrix")
plt.tight_layout()
plt.savefig("output/iris_correlation.png", dpi=120)
plt.show()

### 4.5 Finding 3 - Setosa is linearly separable; Versicolor and Virginica overlap

In [ ]:
fig, ax = plt.subplots()

for species, group in df_iris.groupby("species"):
    ax.scatter(
        group["petal length (cm)"],
        group["petal width (cm)"],
        label=species,
        color=colors[species],
        alpha=0.75,
        s=40,
    )

ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Petal width (cm)")
ax.set_title("Iris: Petal length vs petal width by species")
ax.legend()
plt.tight_layout()
plt.savefig("output/iris_scatter.png", dpi=120)
plt.show()

print("Interpretation:")
print("  Setosa (red) is completely isolated from the other two species.")
print("  Versicolor and Virginica overlap in a narrow band.")
print("  A simple linear classifier will fail on that overlap region.")

In [ ]:
df_iris.to_csv("output/iris.csv", index=False)
print("Saved iris.csv")

---

## Section 5 - Summary of Findings

### Titanic

| Finding | Evidence |
|---|---|
| Survival was class-dependent | 1st class: 63% survived; 3rd class: 24% survived |
| Sex was the strongest predictor | Females survived at 74%, males at 19% |
| Age data is incomplete | 177 rows (19.9%) have no age value |

### Iris

| Finding | Evidence |
|---|---|
| Petal length separates species well | Setosa < 2 cm; Versicolor 3-5 cm; Virginica > 4.5 cm |
| Petal length and width are collinear | Correlation coefficient: 0.96 |
| Setosa is linearly separable | Scatter plot shows zero overlap with other species |

---

**Practice challenge:** Pick any one column from either dataset and write 3 sentences describing what you observe in its distribution.